In [1]:
# ==========================
# 1. Imports
# ==========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
# Detectar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [2]:
class AI4IDataset(Dataset):
    def __init__(self, features, labels, seq_len=10):
        self.seq_len = seq_len
        self.X_seq, self.y_seq = self.create_sequences(features, labels, seq_len)

    def create_sequences(self, features, labels, seq_len):
        X_seq, y_seq = [], []
        for i in range(len(features) - seq_len + 1):
            X_seq.append(features[i:i+seq_len])
            y_seq.append(labels[i+seq_len-1])  # etiqueta del último paso
        return np.array(X_seq), np.array(y_seq)

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return torch.tensor(self.X_seq[idx], dtype=torch.float32), \
               torch.tensor(self.y_seq[idx], dtype=torch.float32)

In [3]:
# ==========================
# 3. Modelo LSTM
# ==========================
class LSTM_MultiLabel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTM_MultiLabel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])  # último hidden state
        return out

In [4]:
# ==========================
# 4. Cargar y preparar datos
# ==========================
df = pd.read_csv("ai4i2020formatted.csv")

feature_cols = ["Air temperature [K]","Process temperature [K]",
                "Rotational speed [rpm]","Torque [Nm]","Tool wear [min]",
                "Type_H","Type_L","Type_M"]
label_cols = ["TWF","HDF","PWF","OSF","RNF"]

X = df[feature_cols].values
y = df[label_cols].values

# Normalizar features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split train/test manteniendo orden temporal
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Crear datasets con secuencias
seq_len = 10
train_dataset = AI4IDataset(X_train, y_train, seq_len)
test_dataset  = AI4IDataset(X_test, y_test, seq_len)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32)


In [5]:
# ==========================
# 5. Entrenar modelo
# ==========================
input_dim = X.shape[1]
hidden_dim = 64
output_dim = y.shape[1]

model = LSTM_MultiLabel(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/100, Loss: 0.1175
Epoch 2/100, Loss: 0.0446
Epoch 3/100, Loss: 0.0400
Epoch 4/100, Loss: 0.0358
Epoch 5/100, Loss: 0.0315
Epoch 6/100, Loss: 0.0289
Epoch 7/100, Loss: 0.0269
Epoch 8/100, Loss: 0.0250
Epoch 9/100, Loss: 0.0236
Epoch 10/100, Loss: 0.0220
Epoch 11/100, Loss: 0.0217
Epoch 12/100, Loss: 0.0199
Epoch 13/100, Loss: 0.0196
Epoch 14/100, Loss: 0.0187
Epoch 15/100, Loss: 0.0186
Epoch 16/100, Loss: 0.0180
Epoch 17/100, Loss: 0.0173
Epoch 18/100, Loss: 0.0169
Epoch 19/100, Loss: 0.0176
Epoch 20/100, Loss: 0.0163
Epoch 21/100, Loss: 0.0160
Epoch 22/100, Loss: 0.0152
Epoch 23/100, Loss: 0.0158
Epoch 24/100, Loss: 0.0150
Epoch 25/100, Loss: 0.0146
Epoch 26/100, Loss: 0.0139
Epoch 27/100, Loss: 0.0136
Epoch 28/100, Loss: 0.0137
Epoch 29/100, Loss: 0.0134
Epoch 30/100, Loss: 0.0128
Epoch 31/100, Loss: 0.0127
Epoch 32/100, Loss: 0.0129
Epoch 33/100, Loss: 0.0122
Epoch 34/100, Loss: 0.0122
Epoch 35/100, Loss: 0.0115
Epoch 36/100, Loss: 0.0116
Epoch 37/100, Loss: 0.0111
Epoch 38/1

In [6]:
# ==========================
# 6. Evaluación
# 6. Evaluación
# ==========================
model.eval()
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()

        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())
        all_labels.append(y_batch.cpu())

all_probs = torch.cat(all_probs).numpy()
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print("\nEjemplo de probabilidades:\n", all_probs[:5])
print("\nEjemplo de predicciones:\n", all_preds[:5])
print("\nEjemplo de etiquetas reales:\n", all_labels[:5])

# Reporte de métricas
print("\n=== Reporte de clasificación (multi-label) ===")
print(classification_report(all_labels, all_preds, target_names=label_cols, zero_division=0))


Ejemplo de probabilidades:
 [[1.02302145e-07 2.47215876e-12 4.34426511e-07 1.15567418e-15
  1.09989347e-03]
 [2.12329132e-06 3.16021105e-05 6.58262870e-04 6.88957095e-01
  1.21948505e-02]
 [8.20911552e-08 2.98305096e-08 2.79882251e-09 6.30168528e-10
  1.03541344e-04]
 [5.86858141e-06 8.15780004e-06 1.53698042e-04 2.06268451e-05
  5.80746491e-05]
 [1.87136629e-06 6.11076871e-07 1.76234528e-06 3.27907196e-06
  1.77369257e-05]]

Ejemplo de predicciones:
 [[0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Ejemplo de etiquetas reales:
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

=== Reporte de clasificación (multi-label) ===
              precision    recall  f1-score   support

         TWF       0.00      0.00      0.00        10
         HDF       0.00      0.00      0.00         0
         PWF       0.54      0.70      0.61        10
         OSF       0.72      0.62      0.67        21
         RNF       0.00      0.00   

In [7]:
from sklearn.metrics import hamming_loss
import numpy as np

# Aseguramos que son numpy arrays enteros
y_true = all_labels.astype(int)
y_pred = all_preds.astype(int)

def subset_accuracy(y_true, y_pred):
    return (y_true == y_pred).all(axis=1).mean()

def example_based_precision(y_true, y_pred):
    precisions = []
    for yt, yp in zip(y_true, y_pred):
        tp = np.logical_and(yt == 1, yp == 1).sum()
        fp = np.logical_and(yt == 0, yp == 1).sum()
        if tp + fp == 0:
            precisions.append(1.0)
        else:
            precisions.append(tp / (tp + fp))
    return np.mean(precisions)

def example_based_recall(y_true, y_pred):
    recalls = []
    for yt, yp in zip(y_true, y_pred):
        tp = np.logical_and(yt == 1, yp == 1).sum()
        fn = np.logical_and(yt == 1, yp == 0).sum()
        if tp + fn == 0:
            recalls.append(1.0)
        else:
            recalls.append(tp / (tp + fn))
    return np.mean(recalls)

def example_based_f1(y_true, y_pred):
    f1s = []
    for yt, yp in zip(y_true, y_pred):
        tp = np.logical_and(yt == 1, yp == 1).sum()
        fp = np.logical_and(yt == 0, yp == 1).sum()
        fn = np.logical_and(yt == 1, yp == 0).sum()
        if tp == 0 and fp == 0 and fn == 0:
            f1s.append(1.0)
        else:
            p = tp / (tp + fp) if (tp + fp) > 0 else 0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1s.append(2*p*r/(p+r) if (p+r) > 0 else 0)
    return np.mean(f1s)

# Calcular
subset_acc = subset_accuracy(y_true, y_pred)
hamming = hamming_loss(y_true, y_pred)
ex_prec = example_based_precision(y_true, y_pred)
ex_rec = example_based_recall(y_true, y_pred)
ex_f1 = example_based_f1(y_true, y_pred)

print("\n=== Métricas extra (como en el paper) ===")
print(f"Subset accuracy: {subset_acc:.4f}")
print(f"Hamming loss: {hamming:.4f}")
print(f"Example-based precision: {ex_prec:.4f}")
print(f"Example-based recall: {ex_rec:.4f}")
print(f"Example-based F1: {ex_f1:.4f}")



=== Métricas extra (como en el paper) ===
Subset accuracy: 0.9804
Hamming loss: 0.0040
Example-based precision: 0.9907
Example-based recall: 0.9902
Example-based F1: 0.9813


In [9]:
all_preds

array([[0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], shape=(1991, 5), dtype=int32)